In [1]:
import os
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import shutil
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
import base64
from datetime import datetime
import openai

warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

In [2]:
variables = []
with open("00 Variables.txt", "r") as f:
    lines = f.readlines()
    hashes = 1
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            hashes = len(s) - len(s.lstrip("#"))
            comment_indent = (hashes - 1) * 2
            print(" " * comment_indent + "\033[92m" + s + "\033[0m")
        elif "=" in s:
            var = s.split("=", 1)[0].strip()
            variables.append(var)
            try:
                exec(s)
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                value_part = s.split("#", 1)[0].strip()
                comment_part = s[s.find("#"):] if "#" in s else ""
                eval_value = f"{var} = {repr(eval(var))}"
                if comment_part:
                    print(" " * black_indent + eval_value + " \033[38;5;250m" + "  " + comment_part + "\033[0m")
                else:
                    print(" " * black_indent + eval_value)
            except Exception as e:
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                print(" " * black_indent + s)

# Global Settings
  TICKER = '^GSPC'
  START_DATE_GLOBAL = '1998-01-01'
  START_DATE = '2024-12-20'
  END_DATE = '2025-06-30'
  WINDOW_SIZE = 120
  HORIZON_DAYS = 1
  CRASH_PERCENTILE = 0.05
  MAX_WINDOWS = 10000
  DATE_INFO = True
  INDICATOR_SIZE = 'small'   # "small", "large"
  CHART_NUM = 4
  TRANSPARENT = False
  NEWS_QUERY = 'Standard and Poors 500 stock index'
  api_key = 'your-api-key'
  MODEL_NAME = 'gpt-4.1'   # "gpt-4.1", "o3-2025-04-16", "gpt-4.1-nano-2025-04-14"
  EMBEDDING_MODEL = 'text-embedding-3-small'
  RUN_TIMESTAMP = '20250726 1012'   # This will be changed automatically
  N_PREV = 6   # Number of previous windows to provide context for
  USE_IMAGES = True   # If True, send images (multimodal model), else just text
# Path
  ## Layer 1
     L1_RESULTS = 'Results'
     L1_COMB_RESULTS = 'Combination Results'
  ## Layer 2
     L2_DATA = 'Data'
     L2_FIGURES = 'Figures'
     L2_OUTPUT = 'Output'
  ## Layer 3
     L3_FIGURES_CHARTS = 'Generate Charts'
# Output Name
  O

In [51]:
NOW_STR = datetime.now().strftime("%Y%m%d %H%M")
RESULTS_DIR = f"{L1_RESULTS}/{L2_OUTPUT}/{NOW_STR}"
FIG_DIR = f"{L1_RESULTS}/{L2_FIGURES}/{L3_FIGURES_CHARTS}"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

shutil.copy("00 Variables.txt", os.path.join(RESULTS_DIR, "Variables.txt"))

'Results/Output/20250726 1709/Variables.txt'

## Class and Definition

In [52]:
class OpenAIAnomalyDetector:
    def __init__(self, api_key: str, model_name: str = "gpt-4.1"):
        openai.api_key = api_key
        self.model_name = model_name
        self.openai = openai

    def detect_anomaly(self, image_paths: list, technical_summary: str = "",
                       news: str = "", retries: int = 2, log_fail_path="gpt41_results/llm_anomaly_failures.txt") -> dict:
        prompt = (
            "You are a financial anomaly detection assistant.\n"
            "Analyze the attached chart, context, technical summary, and recent news.\n"
            "Identify if there is an *anomaly* (e.g., unusual move, technical break, extreme volatility, or other outlier event) in the most recent data.\n"
            "Compare this window with the recent historical pattern.\n"
            "Respond with valid JSON: anomaly_label (1=anomaly, 0=normal), anomaly_score (probability/confidence 0-1), anomaly_type, key_signals, and a brief reasoning."
        )
        for attempt in range(retries):
            try:
                image_contents = []
                for path in image_paths:
                    with open(path, "rb") as image_file:
                        b64_image = base64.b64encode(image_file.read()).decode('utf-8')
                        image_contents.append({
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{b64_image}"}
                        })
                    
                response = self.openai.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": "You are a financial analysis assistant. Never give investment advice."},
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "text", "text": technical_summary or 'N/A'}
                            ] + image_contents
                        }
                    ],
                    # max_tokens=384,
                    temperature=0.2,
                )
                response_text = response.choices[0].message.content
                if not response_text:
                    return {"success": False, "error": "No response from model."}
                response_text = response_text.strip('` \n')
                if response_text.startswith('json'):
                    response_text = response_text[4:].strip('` \n')
                try:
                    result = json.loads(response_text)
                    result['success'] = True
                    return result
                except Exception:
                    with open(log_fail_path, "a") as f:
                        f.write(f"\n---\nPrompt: {technical_summary[:100]}...\nResponse:\n{response_text}\n")
                    if attempt < retries - 1:
                        import time; time.sleep(2)
                        continue
                    return {"success": False, "error": f"Could not parse LLM output to JSON. Output was:\n{response_text}"}
            except Exception as e:
                if attempt < retries - 1:
                    import time; time.sleep(2)
                    continue
                return {"success": False, "error": f"Model call failed: {e}"}

In [53]:
def generate_context_section(window_df):
    # Example: return a short textual summary of key indicator stats in the window
    desc = window_df.describe().loc[['mean', 'std']].to_string()
    return f"Summary statistics:\n{desc}"

In [54]:
def generate_technical_summary(window_df):
    # Example: summarize key indicators' recent behavior
    latest = window_df.iloc[-1]
    
    # summary = (
    #     f"Latest volatility: {latest['volatility']:.4f}, "
    #     f"RSI: {latest['rsi']:.2f}, "
    #     f"MACD: {latest['macd']:.4f}, "
    #     f"Max Drawdown (20d): {latest['maxdd_20']:.4f}"
    # )

    summary = ', '.join([f"{col}: {latest[col]}" for col in window_df.columns])
    return summary

In [55]:
def get_window_news(window_df, df_news):
    # Extract news headlines for dates covered in window_df
    dates = window_df.index.intersection(df_news.index)
    news_texts = df_news.loc[dates, "News"].dropna().unique()
    # Join headlines into a single string for context (limit length)
    combined = " ".join(news_texts)
    if len(combined) > 500:
        combined = combined[:500] + "..."
    return combined

In [56]:
def build_anomaly_prompt(cur_summary, prev_summaries, news_context, context_section, horizon_days):
    prev_str = "\n\n".join([f"PREVIOUS_{i+1}:\n{txt}" for i, txt in enumerate(prev_summaries)])
    return (
        f"You are an advanced financial anomaly detection assistant.\n"
        f"Analyze the attached technical chart and summary (CURRENT), and compare with previous {len(prev_summaries)} windows.\n"
        f"Context:\n{context_section}\n\n"
        f"Technical Summary (CURRENT):\n{cur_summary}\n\n"
        f"{prev_str}\n"
        f"Recent News:\n{news_context}\n\n"
        f"For the next {horizon_days} trading days, determine:\n"
        f"- Is there an anomaly, crash, regime shift, or unusual change relative to previous history?\n"
        f"- What type (e.g., crash, volatility spike, outlier, pattern break)?\n"
        f"Respond in JSON with fields: anomaly_label (0/1), anomaly_type (string), anomaly_score (0-1), key_signals (list), reasoning (string)."
    )

### Zero Shot Learning (Output+Embeddings)

In [57]:
def rolling_llm_anomaly_detection(
    df_ind,
    df_news,
    df_crash,
    predictor,
    save_fig_dir: str,
    results_dir: str,
    n_prev=N_PREV,
    indicator_size=INDICATOR_SIZE,
    date_info=DATE_INFO,
    window_size=WINDOW_SIZE,
    horizon_days=HORIZON_DAYS,
    max_windows=MAX_WINDOWS,
    add_embeddings=True,
    openai_embedding_model=EMBEDDING_MODEL,
    chart_num=CHART_NUM
):
    
    os.makedirs(save_fig_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    start_idx_raw = df_ind.index.get_indexer([START_DATE], method="bfill")[0]
    start_idx = start_idx_raw + window_size
    N = min(len(df_ind) - horizon_days, start_idx + max_windows)


    # N = min(len(df_ind) - horizon_days, window_size + max_windows)
    predictions, embeddings = [], []
    
    # tqdm_bar = tqdm(range(window_size + n_prev, N), desc="LLM Anomaly Scoring")
    tqdm_bar = tqdm(range(start_idx, N), desc="LLM Anomaly Scoring")
    for i in tqdm_bar:
        window = df_ind.iloc[i - window_size: i]
        prev_windows = [df_ind.iloc[i - window_size - j : i - j] for j in range(1, n_prev+1)]
        future = df_ind.iloc[i: i + horizon_days]
        date_str = window.index[-1].strftime("%Y-%m-%d")

        cur_image_path = [f"{save_fig_dir}/Chart {no+1}/{TICKER}_{date_str}_chart{no+1}.png" for no in range(chart_num)]
        curr_b64s = []
        for curr in cur_image_path:
            with open(curr, "rb") as f:
                curr_b64s.append(base64.b64encode(f.read()).decode("utf-8"))

        ### NOTE: REMOVE? ###
        # prev_image_paths = [f"{save_fig_dir}/Chart {no+1}/{TICKER}_{w.index[-1].strftime('%Y-%m-%d')}.png" for no in range(chart_num) for w in prev_windows]
        # prev_b64s = []
        # for p in prev_image_paths:
        #     with open(p, "rb") as f:
        #         prev_b64s.append(base64.b64encode(f.read()).decode("utf-8"))

        cur_summary = generate_technical_summary(window)
        prev_summaries = [generate_technical_summary(w) for w in prev_windows]
        context_section = generate_context_section(window)
        news_context = get_window_news(window, df_news)
        prompt = build_anomaly_prompt(cur_summary, prev_summaries, news_context, context_section, horizon_days)

        # LLM call
        if USE_IMAGES:
            images_content = (
                [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}} for b64 in curr_b64s] # +      # NOTE: REMOVE?
                # [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}} for b64 in prev_b64s]        # NOTE: REMOVE?
            )
            user_content = [{"type": "text", "text": prompt}] + images_content
        else:
            user_content = [{"type": "text", "text": prompt}]

        # Send request
        for attempt in range(2):
            try:
                kwargs = {
                    "model": predictor.model_name,
                    "messages": [
                        {"role": "system", "content": "You are a financial anomaly detection assistant."},
                        {"role": "user", "content": user_content}
                    ],
                    # "max_tokens": 384,
                }
                if predictor.model_name != "o3-2025-04-16":
                    kwargs["temperature"] = 0.2

                response = predictor.openai.chat.completions.create(**kwargs)

                response_text = response.choices[0].message.content
                response_text = response_text.strip('` \n')
                if response_text.startswith('json'):
                    response_text = response_text[4:].strip('` \n')
                result = json.loads(response_text)
                result['success'] = True
                break
            except Exception as e:
                if attempt == 0:
                    import time; time.sleep(2)
                    continue
                result = {"success": False, "error": str(e)}

        tqdm_bar.set_postfix_str(f"{date_str} | {result}")        
        # print(result)
        
        # Collect embeddings if wanted
        if add_embeddings:
            try:
                emb_resp = predictor.openai.embeddings.create(
                    input=str(result), model=openai_embedding_model
                )
                text_embedding = emb_resp.data[0].embedding
            except Exception as e:
                print(f"[WARN] Embedding error at {date_str}: {e}")
                text_embedding = [np.nan] * 1536
        else:
            text_embedding = []
        row = {
            "date": date_str,
            "image_path": cur_image_path,
            "technical_summary": cur_summary,
            "context": context_section,
            "news": news_context,
            "future_dates": ";".join([str(d) for d in future.index]),
            **result
        }
        predictions.append(row)
        if add_embeddings:
            embeddings.append(text_embedding)
            emb_array = np.array(embeddings)
            emb_path = os.path.join(results_dir, f"{LLM_EMBEDDINGS}.npy")
            np.save(emb_path, emb_array)

        results_df = pd.DataFrame(predictions)
        if df_crash is not None:
            _crash = df_crash.rename(columns={'crash_label': 'crash_actual', 'min_future_logret': 'future_min_logret'})
            _crash = _crash.reset_index() if _crash.index.name == 'date' else _crash
            results_df['date'] = pd.to_datetime(results_df['date'])
            _crash['date'] = pd.to_datetime(_crash['date'])
            results_df = results_df.merge(_crash, how='left', on='date')
        out_csv = os.path.join(results_dir, f"{LLM_ANOMALY}.csv")
        results_df.to_csv(out_csv, index=False)
        
    return results_df

### Few Shot Learning (Output+Embeddings)

In [58]:
def rolling_few_shot_llm_anomaly_detection(
    df_ind, 
    df_news, 
    df_crash, 
    detector,
    results_df,
    openai_api_key,
    save_fig_dir: str,
    results_dir=RESULTS_DIR,
    model_name=MODEL_NAME,
    openai_embedding_model=EMBEDDING_MODEL,
    n_pos_fs=2,
    n_neg_fs=2,
    add_embeddings=True,
    date_info=DATE_INFO,
    chart_num=CHART_NUM,
):

    openai.api_key = openai_api_key
    all_results, embeddings = [], []

    for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Few-Shot LLM Anomaly"):
        date_str = row['date'].strftime("%Y-%m-%d")
        # Get all previous labeled rows—**no lookahead!**

        mask_days = 4 # NOTE: WHY 4?
        if idx<mask_days:
            prev = results_df.iloc[:idx]
        else:
            prev = results_df.iloc[:idx-mask_days]

        pos_fs = prev[prev['crash_actual'] == 1].tail(n_pos_fs)
        neg_fs = prev[prev['crash_actual'] == 0].tail(n_neg_fs)

        # Compose few-shot prompt
        fewshot = []
        for ex in pd.concat([pos_fs, neg_fs]).itertuples():
            fs_context = (
                f"Chart: {getattr(ex, 'image_path', 'N/A') if date_info else 'N/A'}\n"
                f"Context: {getattr(ex, 'context', 'N/A')}\n"
                f"Crash Actual: {getattr(ex, 'crash_actual', 'N/A')}\n"
                f"Technical Summary: {str(getattr(ex, 'technical_summary', ''))[:]}...\n"
            )
            fewshot.append(fs_context)
            
        prompt = (
            "You are an anomaly detection assistant for financial market time series. "
            "Here are examples of previous windows and their anomaly labels:\n"
            + "\n\n".join(fewshot) +
            f"\n\nNow analyze the new chart below. Respond in JSON with the following fields: "
            f"anomaly_label (1=anomaly, 0=normal), anomaly_score (0-1), anomaly_type (e.g., 'crash', 'spike', 'regime_change'), key_signals, reasoning.\n"
            f"Chart: {row['image_path'] if ('image_path' in results_df.columns and date_info) else 'N/A'}\n"
            f"Context: {row['context'] if 'context' in results_df.columns else 'N/A'}\n"
            f"Technical Summary: {str(row['technical_summary'])[:] if 'technical_summary' in results_df.columns else ''}...\n"
            # f"Technical Summary: {str(row['technical_summary'])[:140] if 'technical_summary' in results_df.columns else ''}..."
            f"**STRICTLY RESPOND IN THE FOLLOWING JSON FORMAT**\n"
            f"anomaly_label (1=anomaly, 0=normal), anomaly_score (0-1), anomaly_type (e.g., 'crash', 'spike', 'regime_change'), key_signals, reasoning.\n"
        )
        
        # Call LLM
        try:
            cur_image_path = [f"{save_fig_dir}/Chart {no+1}/{TICKER}_{date_str}_chart{no+1}.png" for no in range(chart_num)]
            curr_b64s = []
            for curr in cur_image_path:
                with open(curr, "rb") as f:
                    curr_b64s.append(base64.b64encode(f.read()).decode("utf-8"))

            if USE_IMAGES:
                images_content = (
                    [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}} for b64 in curr_b64s]
                )
                user_content = [{"type": "text", "text": prompt}] + images_content
            else:
                user_content = [{"type": "text", "text": prompt}]

            messages = [
                {"role": "system", "content": "You are a financial anomaly detection assistant. Never give investment advice."},
                {"role": "user", "content": user_content}
            ]

            kwargs = {
                "model": model_name,
                "messages": messages,
                # "max_tokens": 384,
            }
            if model_name != "o3-2025-04-16":
                kwargs["temperature"] = 0.2

            response = openai.chat.completions.create(**kwargs)
            response_text = response.choices[0].message.content

            # Remove code fencing if present
            if response_text.startswith('```json'): response_text = response_text[7:]
            if response_text.startswith('```'): response_text = response_text[3:]
            response_text = response_text.strip('` \n')
            result = json.loads(response_text)
            result['success'] = True
        except Exception as e:
            result = {"success": False, "error": str(e)}
        # Add identifying fields for analysis
        result.update({
            "date": row['date'],
            "image_path": row['image_path'] if 'image_path' in results_df.columns else None,
            "context": row['context'] if 'context' in results_df.columns else None,
            "technical_summary": row['technical_summary'] if 'technical_summary' in results_df.columns else None,
            "crash_actual": row['crash_actual'] if 'crash_actual' in results_df.columns else None,
        })
        all_results.append(result)
        
        if add_embeddings:
            try:
                emb_resp = detector.openai.embeddings.create(
                    input=str(result), model=openai_embedding_model
                )
                text_embedding = emb_resp.data[0].embedding
            except Exception as e:
                print(f"[WARN] Embedding error at {date_str}: {e}")
                text_embedding = [np.nan] * 1536
        else:
            text_embedding = []

        if add_embeddings:
            embeddings.append(text_embedding)
            emb_array = np.array(embeddings)
            emb_path = os.path.join(results_dir, f"{LLM_EMBEDDINGS_FEWSHOT}.npy")
            np.save(emb_path, emb_array)
        
        fewshot_df = pd.DataFrame(all_results)
        out_csv = os.path.join(results_dir, f"{LLM_ANOMALY_FEWSHOT}.csv")
        fewshot_df.to_csv(out_csv, index=False)
            
    if df_crash is not None:
        _crash = df_crash.rename(columns={'crash_label': 'crash_actual', 'min_future_logret': 'future_min_logret'})
        _crash = _crash.reset_index() if _crash.index.name == 'date' else _crash
        fewshot_df['date'] = pd.to_datetime(fewshot_df['date'])
        _crash['date'] = pd.to_datetime(_crash['date'])
        fewshot_df = fewshot_df.merge(_crash, how='left', on='date')

    out_csv = os.path.join(results_dir, f"{LLM_ANOMALY_FEWSHOT}.csv")
    fewshot_df.to_csv(out_csv, index=False)
    print(f"Finished saving few-shot anomaly predictions to {out_csv}")

    if add_embeddings:
        print(f"Finished Saving embeddings to {emb_path}")


    return fewshot_df, emb_array

## Main Execution

In [59]:
print(f"Inputing price data for {TICKER} from {START_DATE} to {END_DATE} ...")
df = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv", index_col=0, parse_dates=True)
print("Data inputted. Shape:", df.shape, "\n")

print("Inputing indicators...")
df_ind = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}.csv", index_col=0, parse_dates=True)
print("Indicators inputed. Shape:", df_ind.shape, "\n")

print("Inputing crash labels data...")
df_crash = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{CRASH_LABEL}.csv", index_col=0, parse_dates=True)
print("Crash labels data inputed. Shape:", df_crash.shape, "\n")

print("Inputing news data...")
df_news = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv", index_col=0, parse_dates=True)
print("News data inputed. Shape:", df_news.shape, "\n")

detector = OpenAIAnomalyDetector(api_key, model_name=MODEL_NAME)

Inputing price data for ^GSPC from 2023-07-01 to 2025-06-30 ...
Data inputted. Shape: (6913, 6) 

Inputing indicators...
Indicators inputed. Shape: (6674, 69) 

Inputing crash labels data...
Crash labels data inputed. Shape: (6674, 3) 

Inputing news data...
News data inputed. Shape: (10043, 1) 



In [60]:
anomaly_df = rolling_llm_anomaly_detection(
    df_ind, df_news, df_crash, detector,
    window_size=WINDOW_SIZE,
    horizon_days=HORIZON_DAYS,
    save_fig_dir=FIG_DIR,
    results_dir=RESULTS_DIR,
    max_windows=MAX_WINDOWS,
    openai_embedding_model=EMBEDDING_MODEL
)

print("Done. LLM-based anomaly detection complete.")

LLM Anomaly Scoring: 100%|██████████| 378/378 [1:20:45<00:00, 12.82s/it, 2025-06-25 | {'anomaly_label': 0, 'anomaly_type': 'none', 'anomaly_score': 0.08, 'key_signals': ['Volatility (all measures) stable and near historical mean', 'No crash or spike in returns (return and logreturn near zero)', 'ATR, TR, and volume are not showing abnormal jumps', 'MACD, RSI, Stochastics, CCI all in normal/neutral or slightly overbought territory', 'No regime shift in SMA20/50/200 or Bollinger Bands', 'No outlier in VaR/ES, skewness, or kurtosis', 'ADX is low, indicating weak trend strength'], 'reasoning': 'The current session shows a flat return after a recent rally, with volatility measures (realized, historical, Parkinson, etc.) all stable and close to their recent averages. No crash or spike is present in the return distribution, and the price is near the upper Bollinger Band but not breaking out. Oscillators (RSI, Stoch, MACD) are in neutral to slightly overbought territory, but not extreme. Volum

Done. LLM-based anomaly detection complete.


In [61]:
output_path = os.path.join(RESULTS_DIR, f"{LLM_ANOMALY}.csv")
output = pd.read_csv(output_path)

path = os.path.join(RESULTS_DIR, f"{LLM_EMBEDDINGS}.npy")
data = np.load(path)


# Now `data` is a NumPy array
display(data)
display(output)

array([[-0.0055419 ,  0.04600283, -0.01010625, ...,  0.01476403,
         0.0134127 , -0.02179384],
       [ 0.00834902,  0.03680327,  0.00230724, ...,  0.00712764,
         0.00741275, -0.02694776],
       [ 0.00610524,  0.02187015, -0.00295853, ...,  0.01855269,
         0.0072552 , -0.0274039 ],
       ...,
       [-0.03009373,  0.04525416, -0.00826158, ...,  0.00814092,
         0.01462101, -0.01244915],
       [-0.00619428,  0.04036578, -0.01413669, ..., -0.00359557,
        -0.00435586, -0.00809049],
       [-0.02305251,  0.05734159, -0.01269841, ...,  0.00112501,
         0.00012199, -0.01817524]])

,date,image_path,technical_summary,context,news,future_dates,anomaly_label,anomaly_type,anomaly_score,key_signals,reasoning,success,crash_actual,future_min_logret,crash_threshold
0,2023-12-20,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 4698.35009765625, high: 4778.009765625,...",Summary statistics:\n close ...,Top 25 Stocks in the S&P 500 - Investopedia Eq...,2023-12-21 00:00:00,1,pattern break / overbought reversal,0.72,['Sharp negative return after persistent uptre...,The S&P 500 experienced a sharp reversal after...,True,0.0,0.010249,-0.013493
1,2023-12-21,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 4746.75, high: 4748.7099609375, low: 47...",Summary statistics:\n close ...,Industry Analysts Project 9% Increase in S&P 5...,2023-12-22 00:00:00,1,"pattern break / regime shift (momentum spike, ...",0.82,"['Close at new high, well above SMA20/SMA50/SM...","The S&P 500 has broken out to new highs, with ...",True,0.0,0.001659,-0.013564
2,2023-12-22,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 4754.6298828125, high: 4772.93994140625...",Summary statistics:\n close ...,Industry Analysts Project 9% Increase in S&P 5...,2023-12-26 00:00:00,1,Overbought/Regime Shift,0.82,"['RSI > 75 (75.2, overbought)', 'Stoch K > 89 ...",The current S&P 500 price is at or near its 20...,True,0.0,0.004223,-0.013564
3,2023-12-26,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 4774.75, high: 4784.72021484375, low: 4...",Summary statistics:\n close ...,Industry Analysts Project 9% Increase in S&P 5...,2023-12-27 00:00:00,1,pattern break / overbought regime,0.82,"['RSI at 77 (strongly overbought, >70)', 'Stoc...","The S&P 500 is exhibiting a strong, persistent...",True,0.0,0.001429,-0.013564
4,2023-12-27,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 4781.580078125, high: 4785.39013671875,...",Summary statistics:\n close ...,Market Weight vs. Equal Weight S&P 500 ETFs: W...,2023-12-28 00:00:00,1,overbought_extreme_momentum,0.82,"['RSI > 80 (81.1, persistent over 70 for sever...",The S&P 500 is exhibiting a regime of extreme ...,True,0.0,0.000370,-0.013564
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2025-06-18,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 5980.8701171875, high: 6018.25, low: 59...",Summary statistics:\n close ...,"Stock market today: Nasdaq, S&P 500 jump as Nv...",2025-06-20 00:00:00,0,none,0.13,"['Volatility (composite, realized, Parkinson, ...","Across the last 6 windows, the current period ...",True,0.0,-0.002181,-0.020071
374,2025-06-20,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 5967.83984375, high: 6018.2001953125, l...",Summary statistics:\n close ...,Markets stumble as Wall Street sells off Big T...,2025-06-23 00:00:00,0,none,0.13,"['Volatility (composite, HV_10, HV_30) stable ...",The current window shows a mild negative retur...,True,0.0,0.009561,-0.020071
375,2025-06-23,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 6025.169921875, high: 6028.77001953125,...",Summary statistics:\n close ...,Markets stumble as Wall Street sells off Big T...,2025-06-24 00:00:00,0,none,0.13,['Volatility (0.2456) and composite_vol stable...,The current session shows a moderate positive ...,True,0.0,0.011060,-0.020071
376,2025-06-24,['Results/Figures/Generate Charts/Chart 1/^GSP...,"close: 6092.18017578125, high: 6101.759765625,...",Summary statistics:\n close ...,Stocks just did something they haven’t done in...,2025-06-25 00:00:00,1,pattern break / regime shift,0.82,"['Close at new 20-day high (6092.18), breaking...",The current session shows a clear pattern brea...,True,0.0,-0.000003,-0.020071


In [62]:
# RUN_TIMESTAMP = "20250720 1721"
# LLM_ANOMALY = "Rolling_LLM_Anomaly_Scores"
# LLM_ANOMALY_FEWSHOT = "Rolling_LLM_Anomaly_Scores_FewShot"
# LLM_EMBEDDINGS = "LLM_Anomaly_Embeddings"
# LLM_EMBEDDINGS_FEWSHOT = "LLM_Anomaly_Embeddings_FewShot"
# RESULTS_DIR = f"{L1_RESULTS}/{L2_OUTPUT}/{RUN_TIMESTAMP} {MODEL_NAME}"

In [63]:
if os.path.exists(output_path):
    print(f"Loading window-level anomaly results from: {output_path}")
    results_df = pd.read_csv(output_path)
    results_df['date'] = pd.to_datetime(results_df['date'])
    # Normalize columns: strip and lowercase for consistent access
    results_df.columns = [col.strip().lower() for col in results_df.columns]
    print("Columns in results_df:", list(results_df.columns))
    if 'crash_actual' not in results_df.columns:
        print("[WARN] 'crash_actual' column not found in results_df. Filling with 0.")
        results_df['crash_actual'] = 0
    else:
        # Ensure no NaNs in crash_actual and convert to int
        results_df['crash_actual'] = results_df['crash_actual'].fillna(0).astype(int)
else:
    raise FileNotFoundError("No window-level anomaly results found. Run rolling_openai_anomaly_detection first.")


# ==== Run Few-Shot Anomaly Detection with embedding ====
fewshot_path = os.path.join(RESULTS_DIR, "fewshot_rolling_anomaly_predictions.csv")
embedding_path = os.path.join(RESULTS_DIR, "fewshot_llm_anomaly_embeddings.npy")

fewshot_results, emb_array = rolling_few_shot_llm_anomaly_detection(
    df_ind, df_news, df_crash, detector,
    date_info=DATE_INFO,
    save_fig_dir=FIG_DIR,
    results_dir=RESULTS_DIR,
    results_df=results_df,
    openai_api_key=api_key,
    model_name=MODEL_NAME,
    openai_embedding_model=EMBEDDING_MODEL,
    n_pos_fs=2,
    n_neg_fs=2,
    add_embeddings=True, 
)

print("\nFew-shot LLM anomaly detection done.")
display(fewshot_results)

Loading window-level anomaly results from: Results/Output/20250726 1709/Rolling_LLM_Anomaly_Scores.csv
Columns in results_df: ['date', 'image_path', 'technical_summary', 'context', 'news', 'future_dates', 'anomaly_label', 'anomaly_type', 'anomaly_score', 'key_signals', 'reasoning', 'success', 'crash_actual', 'future_min_logret', 'crash_threshold']


Few-Shot LLM Anomaly: 100%|██████████| 378/378 [1:15:10<00:00, 11.93s/it]

Finished saving few-shot anomaly predictions to Results/Output/20250726 1709/Rolling_LLM_Anomaly_Scores_FewShot.csv
Finished Saving embeddings to Results/Output/20250726 1709/LLM_Anomaly_Embeddings_FewShot.npy

Few-shot LLM anomaly detection done.


,anomaly_label,anomaly_score,anomaly_type,key_signals,reasoning,success,date,image_path,context,technical_summary,crash_actual_x,error,crash_actual_y,future_min_logret,crash_threshold
0,1.0,0.82,regime_change,[Price breakout above upper Bollinger Band and...,The S&P 500 has experienced a sharp upward mov...,True,2023-12-20,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 4698.35009765625, high: 4778.009765625,...",0,NaN,0.0,0.010249,-0.013493
1,0.0,0.18,normal,"{'price_trend': 'Strong uptrend, price near up...",The market is exhibiting a strong bullish tren...,True,2023-12-21,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 4746.75, high: 4748.7099609375, low: 47...",0,NaN,0.0,0.001659,-0.013564
2,0.0,0.15,normal,"{'price_trend': 'Strong uptrend, price near up...","The market is exhibiting a strong, persistent ...",True,2023-12-22,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 4754.6298828125, high: 4772.93994140625...",0,NaN,0.0,0.004223,-0.013564
3,0.0,0.13,normal,"{'price': 'New highs, close at 4774.75, above ...","Despite strong overbought signals (RSI, Stoch,...",True,2023-12-26,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 4774.75, high: 4784.72021484375, low: 4...",0,NaN,0.0,0.001429,-0.013564
4,1.0,0.87,regime_change,"[RSI extremely overbought (81.1), Stochastic %...","The S&P 500 is exhibiting highly unusual, pers...",True,2023-12-27,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 4781.580078125, high: 4785.39013671875,...",0,NaN,0.0,0.000370,-0.013564
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,0.0,0.13,normal,{'price_trend': 'Price is above SMA20 and SMA5...,The current window shows a market that has sta...,True,2025-06-18,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 5980.8701171875, high: 6018.25, low: 59...",0,NaN,0.0,-0.002181,-0.020071
374,0.0,0.18,normal,"{'volatility': 0.245, 'hv_10': 0.112, 'hv_30':...",The current window does not show signs of a cr...,True,2025-06-20,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 5967.83984375, high: 6018.2001953125, l...",0,NaN,0.0,0.009561,-0.020071
375,0.0,0.15,normal,"{'return': 0.00956, 'volatility': 0.2456, 'hv_...",The current window shows a strong positive ret...,True,2025-06-23,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 6025.169921875, high: 6028.77001953125,...",0,NaN,0.0,0.011060,-0.020071
376,1.0,0.85,spike,"{'close': 6092.18, 'return': 0.01106, 'zscore'...","The close price has surged to a new high, brea...",True,2025-06-24,['Results/Figures/Generate Charts/Chart 1/^GSP...,Summary statistics:\n close ...,"close: 6092.18017578125, high: 6101.759765625,...",0,NaN,0.0,-0.000003,-0.020071
